# Black Grouse 2030 Land-Use Scenario Project

## Stage 6: Simulating Alternative 2030 Restoration Scenarios

This notebook creates four alternative 2030 restoration scenarios:

1. Dispersed restoration
2. Patch enlargement
3. Connectivity-focused restoration
4. Integrated low-matrix-pressure restoration

Each scenario converts exactly the same area of agricultural matrix into core habitat. This allows differences in habitat configuration to be compared independently of restored habitat amount.

The restoration budget is based on the observed net increase in core habitat between 2012 and 2024: 18,649 pixels, equivalent to approximately 11.66 km².

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio

from scipy import ndimage

print("Packages imported successfully.")

Packages imported successfully.


In [2]:
PROJECT_DIR = Path.cwd()

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
TABLES_DIR = PROJECT_DIR / "outputs" / "tables"

LANDUSE_2024_PATH = (
    PROCESSED_DIR / "LGN2024_harmonised_25m.tif"
)

SUITABILITY_PATH = (
    PROCESSED_DIR / "restoration_suitability_score.tif"
)

DISTANCE_TO_CORE_PATH = (
    PROCESSED_DIR / "distance_to_core_habitat_2024_m.tif"
)

DISTANCE_TO_URBAN_PATH = (
    PROCESSED_DIR / "distance_to_urban_2024_m.tif"
)

SURROUNDING_CORE_PATH = (
    PROCESSED_DIR
    / "surrounding_core_habitat_1km_proportion.tif"
)

SURROUNDING_URBAN_PATH = (
    PROCESSED_DIR
    / "surrounding_urban_1km_proportion.tif"
)

AGRICULTURAL_CLASS = 1
CORE_HABITAT_CLASS = 2

RESTORATION_TARGET_PIXELS = 18_649

PIXEL_SIZE_METRES = 25
PIXEL_AREA_KM2 = (
    PIXEL_SIZE_METRES
    * PIXEL_SIZE_METRES
    / 1_000_000
)

RESTORATION_TARGET_AREA_KM2 = (
    RESTORATION_TARGET_PIXELS
    * PIXEL_AREA_KM2
)

RANDOM_SEED = 42

print(
    "Restoration target:",
    f"{RESTORATION_TARGET_PIXELS:,} pixels",
)

print(
    "Restoration target area:",
    f"{RESTORATION_TARGET_AREA_KM2:.3f} km²",
)

Restoration target: 18,649 pixels
Restoration target area: 11.656 km²


In [3]:
def read_float_raster(
    raster_path,
):
    """
    Read a floating-point raster and convert NoData values
    to NaN.
    """

    with rasterio.open(raster_path) as src:

        raster_data = src.read(1).astype(np.float32)

        if src.nodata is not None:
            raster_data[
                raster_data == src.nodata
            ] = np.nan

        profile = src.profile.copy()
        transform = src.transform
        crs = src.crs

    return raster_data, profile, transform, crs


with rasterio.open(LANDUSE_2024_PATH) as src:

    landuse_2024 = src.read(1)

    landuse_profile = src.profile.copy()
    reference_transform = src.transform
    reference_crs = src.crs
    reference_shape = landuse_2024.shape


restoration_suitability, _, suitability_transform, suitability_crs = (
    read_float_raster(SUITABILITY_PATH)
)

distance_to_core, _, core_transform, core_crs = (
    read_float_raster(DISTANCE_TO_CORE_PATH)
)

distance_to_urban, _, urban_transform, urban_crs = (
    read_float_raster(DISTANCE_TO_URBAN_PATH)
)

surrounding_core, _, surrounding_core_transform, surrounding_core_crs = (
    read_float_raster(SURROUNDING_CORE_PATH)
)

surrounding_urban, _, surrounding_urban_transform, surrounding_urban_crs = (
    read_float_raster(SURROUNDING_URBAN_PATH)
)


input_arrays = {
    "restoration_suitability": restoration_suitability,
    "distance_to_core": distance_to_core,
    "distance_to_urban": distance_to_urban,
    "surrounding_core": surrounding_core,
    "surrounding_urban": surrounding_urban,
}

input_transforms = {
    "restoration_suitability": suitability_transform,
    "distance_to_core": core_transform,
    "distance_to_urban": urban_transform,
    "surrounding_core": surrounding_core_transform,
    "surrounding_urban": surrounding_urban_transform,
}

input_crs = {
    "restoration_suitability": suitability_crs,
    "distance_to_core": core_crs,
    "distance_to_urban": urban_crs,
    "surrounding_core": surrounding_core_crs,
    "surrounding_urban": surrounding_urban_crs,
}


for raster_name, raster_array in input_arrays.items():

    if raster_array.shape != reference_shape:
        raise ValueError(
            f"{raster_name} has a different raster shape."
        )

    if input_transforms[raster_name] != reference_transform:
        raise ValueError(
            f"{raster_name} has a different transform."
        )

    if input_crs[raster_name] != reference_crs:
        raise ValueError(
            f"{raster_name} has a different CRS."
        )


analysis_mask = landuse_2024 != 0
candidate_mask = landuse_2024 == AGRICULTURAL_CLASS
existing_core_mask = landuse_2024 == CORE_HABITAT_CLASS


candidate_pixels = int(candidate_mask.sum())

if candidate_pixels < RESTORATION_TARGET_PIXELS:
    raise ValueError(
        "There are not enough eligible candidate pixels "
        "for the selected restoration target."
    )


print("All simulation inputs are aligned.")

print(
    "Eligible candidate pixels:",
    f"{candidate_pixels:,}",
)

print(
    "Eligible candidate area:",
    f"{candidate_pixels * PIXEL_AREA_KM2:.2f} km²",
)

print(
    "Existing core-habitat area:",
    f"{existing_core_mask.sum() * PIXEL_AREA_KM2:.2f} km²",
)

All simulation inputs are aligned.
Eligible candidate pixels: 730,030
Eligible candidate area: 456.27 km²
Existing core-habitat area: 54.41 km²


In [4]:
def scale_candidate_values(
    raster_array,
    candidate_pixels,
    higher_is_better=True,
):
    """
    Scale candidate-pixel values between 0 and 1 using
    the 2nd and 98th percentiles.
    """

    finite_candidates = (
        candidate_pixels
        & np.isfinite(raster_array)
    )

    values = raster_array[finite_candidates]

    lower_limit = np.percentile(values, 2)
    upper_limit = np.percentile(values, 98)

    if upper_limit == lower_limit:
        scaled = np.zeros(
            raster_array.shape,
            dtype=np.float32,
        )
    else:
        scaled = (
            raster_array - lower_limit
        ) / (
            upper_limit - lower_limit
        )

    scaled = np.clip(
        scaled,
        0,
        1,
    )

    if not higher_is_better:
        scaled = 1 - scaled

    scaled[~candidate_pixels] = np.nan

    return scaled.astype(np.float32)


def select_highest_scoring_pixels(
    score_array,
    candidate_pixels,
    number_of_pixels,
):
    """
    Select exactly the requested number of candidate pixels
    with the highest scores.
    """

    candidate_indices = np.flatnonzero(
        candidate_pixels
    )

    candidate_scores = score_array.ravel()[
        candidate_indices
    ]

    candidate_scores = np.where(
        np.isfinite(candidate_scores),
        candidate_scores,
        -np.inf,
    )

    order = np.argsort(
        candidate_scores,
        kind="stable",
    )[::-1]

    selected_indices = candidate_indices[
        order[:number_of_pixels]
    ]

    selected_mask = np.zeros(
        candidate_pixels.shape,
        dtype=bool,
    )

    selected_mask.ravel()[
        selected_indices
    ] = True

    return selected_mask


def create_scenario_raster(
    baseline_landuse,
    selected_restoration_pixels,
):
    """
    Convert selected agricultural pixels to core habitat.
    """

    scenario = baseline_landuse.copy()

    scenario[
        selected_restoration_pixels
    ] = CORE_HABITAT_CLASS

    return scenario

In [5]:
rng = np.random.default_rng(
    RANDOM_SEED
)


# ---------------------------------------------------------
# 1. Dispersed restoration
# ---------------------------------------------------------

candidate_indices = np.flatnonzero(
    candidate_mask
)

dispersed_indices = rng.choice(
    candidate_indices,
    size=RESTORATION_TARGET_PIXELS,
    replace=False,
)

dispersed_mask = np.zeros(
    candidate_mask.shape,
    dtype=bool,
)

dispersed_mask.ravel()[
    dispersed_indices
] = True


# ---------------------------------------------------------
# 2. Patch-enlargement restoration
# ---------------------------------------------------------

core_proximity_scaled = scale_candidate_values(
    distance_to_core,
    candidate_mask,
    higher_is_better=False,
)

patch_enlargement_score = (
    0.95 * core_proximity_scaled
    + 0.05 * restoration_suitability
)

patch_enlargement_mask = (
    select_highest_scoring_pixels(
        score_array=patch_enlargement_score,
        candidate_pixels=candidate_mask,
        number_of_pixels=RESTORATION_TARGET_PIXELS,
    )
)


# ---------------------------------------------------------
# 3. Connectivity-focused restoration
# ---------------------------------------------------------

CONNECTIVITY_GAP_RADIUS_METRES = 1000

connectivity_iterations = int(
    CONNECTIVITY_GAP_RADIUS_METRES
    / PIXEL_SIZE_METRES
)

connectivity_structure = np.ones(
    (3, 3),
    dtype=bool,
)

closed_core_network = ndimage.binary_closing(
    existing_core_mask,
    structure=connectivity_structure,
    iterations=connectivity_iterations,
)

potential_bridge_zone = (
    closed_core_network
    & candidate_mask
)

surrounding_core_scaled = scale_candidate_values(
    surrounding_core,
    candidate_mask,
    higher_is_better=True,
)

connectivity_score = (
    2.0 * potential_bridge_zone.astype(np.float32)
    + 0.60 * surrounding_core_scaled
    + 0.40 * core_proximity_scaled
)

connectivity_mask = (
    select_highest_scoring_pixels(
        score_array=connectivity_score,
        candidate_pixels=candidate_mask,
        number_of_pixels=RESTORATION_TARGET_PIXELS,
    )
)


# ---------------------------------------------------------
# 4. Integrated low-matrix-pressure restoration
# ---------------------------------------------------------

integrated_mask = (
    select_highest_scoring_pixels(
        score_array=restoration_suitability,
        candidate_pixels=candidate_mask,
        number_of_pixels=RESTORATION_TARGET_PIXELS,
    )
)


RESTORATION_MASKS = {
    "dispersed": dispersed_mask,
    "patch_enlargement": patch_enlargement_mask,
    "connectivity_focused": connectivity_mask,
    "integrated_low_matrix_pressure": integrated_mask,
}


for scenario_name, restoration_mask in RESTORATION_MASKS.items():

    selected_pixels = int(
        restoration_mask.sum()
    )

    selected_area_km2 = (
        selected_pixels
        * PIXEL_AREA_KM2
    )

    invalid_pixels = int(
        np.sum(
            restoration_mask
            & ~candidate_mask
        )
    )

    print(f"\nScenario: {scenario_name}")
    print(f"Selected pixels: {selected_pixels:,}")
    print(
        "Restored area:",
        f"{selected_area_km2:.3f} km²",
    )
    print(
        "Invalid selected pixels:",
        invalid_pixels,
    )


print(
    "\nCandidate pixels inside potential bridge zone:",
    f"{potential_bridge_zone.sum():,}",
)


Scenario: dispersed
Selected pixels: 18,649
Restored area: 11.656 km²
Invalid selected pixels: 0

Scenario: patch_enlargement
Selected pixels: 18,649
Restored area: 11.656 km²
Invalid selected pixels: 0

Scenario: connectivity_focused
Selected pixels: 18,649
Restored area: 11.656 km²
Invalid selected pixels: 0

Scenario: integrated_low_matrix_pressure
Selected pixels: 18,649
Restored area: 11.656 km²
Invalid selected pixels: 0

Candidate pixels inside potential bridge zone: 580,959


In [6]:
SCENARIO_RASTERS = {}
RESTORATION_MASK_PATHS = {}

scenario_profile = landuse_profile.copy()
scenario_profile.update(
    {
        "driver": "GTiff",
        "dtype": "uint8",
        "count": 1,
        "nodata": 0,
        "compress": "lzw",
    }
)

for scenario_name, restoration_mask in RESTORATION_MASKS.items():

    scenario_raster = create_scenario_raster(
        baseline_landuse=landuse_2024,
        selected_restoration_pixels=restoration_mask,
    )

    scenario_output_path = (
        PROCESSED_DIR
        / f"landuse_2030_{scenario_name}.tif"
    )

    mask_output_path = (
        PROCESSED_DIR
        / f"restoration_mask_{scenario_name}.tif"
    )

    with rasterio.open(
        scenario_output_path,
        "w",
        **scenario_profile,
    ) as destination:
        destination.write(
            scenario_raster.astype(np.uint8),
            1,
        )

    mask_profile = scenario_profile.copy()

    with rasterio.open(
        mask_output_path,
        "w",
        **mask_profile,
    ) as destination:
        destination.write(
            restoration_mask.astype(np.uint8),
            1,
        )

    SCENARIO_RASTERS[scenario_name] = scenario_output_path
    RESTORATION_MASK_PATHS[scenario_name] = mask_output_path

    print(f"\nSaved scenario: {scenario_output_path}")
    print(f"Saved restoration mask: {mask_output_path}")


Saved scenario: C:\Users\smit1\BlackGrouse_2030\data\processed\landuse_2030_dispersed.tif
Saved restoration mask: C:\Users\smit1\BlackGrouse_2030\data\processed\restoration_mask_dispersed.tif

Saved scenario: C:\Users\smit1\BlackGrouse_2030\data\processed\landuse_2030_patch_enlargement.tif
Saved restoration mask: C:\Users\smit1\BlackGrouse_2030\data\processed\restoration_mask_patch_enlargement.tif

Saved scenario: C:\Users\smit1\BlackGrouse_2030\data\processed\landuse_2030_connectivity_focused.tif
Saved restoration mask: C:\Users\smit1\BlackGrouse_2030\data\processed\restoration_mask_connectivity_focused.tif

Saved scenario: C:\Users\smit1\BlackGrouse_2030\data\processed\landuse_2030_integrated_low_matrix_pressure.tif
Saved restoration mask: C:\Users\smit1\BlackGrouse_2030\data\processed\restoration_mask_integrated_low_matrix_pressure.tif


In [7]:
scenario_validation_records = []

baseline_core_pixels = int(
    np.sum(landuse_2024 == CORE_HABITAT_CLASS)
)

for scenario_name, scenario_path in SCENARIO_RASTERS.items():

    with rasterio.open(scenario_path) as src:
        scenario_data = src.read(1)

    changed_mask = scenario_data != landuse_2024

    agricultural_to_core = (
        (landuse_2024 == AGRICULTURAL_CLASS)
        & (scenario_data == CORE_HABITAT_CLASS)
    )

    invalid_changes = (
        changed_mask
        & ~agricultural_to_core
    )

    restored_pixels = int(
        agricultural_to_core.sum()
    )

    scenario_core_pixels = int(
        np.sum(
            scenario_data == CORE_HABITAT_CLASS
        )
    )

    selected_mask = RESTORATION_MASKS[
        scenario_name
    ]

    scenario_validation_records.append(
        {
            "scenario": scenario_name,
            "restored_pixels": restored_pixels,
            "restored_area_km2": round(
                restored_pixels * PIXEL_AREA_KM2,
                3,
            ),
            "baseline_core_area_km2": round(
                baseline_core_pixels * PIXEL_AREA_KM2,
                3,
            ),
            "scenario_core_area_km2": round(
                scenario_core_pixels * PIXEL_AREA_KM2,
                3,
            ),
            "invalid_changes": int(
                invalid_changes.sum()
            ),
            "mean_distance_to_core_m": round(
                float(
                    np.nanmean(
                        distance_to_core[selected_mask]
                    )
                ),
                2,
            ),
            "mean_distance_to_urban_m": round(
                float(
                    np.nanmean(
                        distance_to_urban[selected_mask]
                    )
                ),
                2,
            ),
            "mean_surrounding_core": round(
                float(
                    np.nanmean(
                        surrounding_core[selected_mask]
                    )
                ),
                4,
            ),
            "mean_surrounding_urban": round(
                float(
                    np.nanmean(
                        surrounding_urban[selected_mask]
                    )
                ),
                4,
            ),
            "mean_suitability_score": round(
                float(
                    np.nanmean(
                        restoration_suitability[
                            selected_mask
                        ]
                    )
                ),
                4,
            ),
        }
    )


scenario_validation = pd.DataFrame(
    scenario_validation_records
)

SCENARIO_SUMMARY_PATH = (
    TABLES_DIR
    / "2030_restoration_scenario_summary.csv"
)

scenario_validation.to_csv(
    SCENARIO_SUMMARY_PATH,
    index=False,
)

display(scenario_validation)


all_checks_passed = (
    (
        scenario_validation["restored_pixels"]
        == RESTORATION_TARGET_PIXELS
    ).all()
    and (
        scenario_validation["invalid_changes"]
        == 0
    ).all()
    and (
        scenario_validation["scenario_core_area_km2"]
        == round(
            (
                baseline_core_pixels
                + RESTORATION_TARGET_PIXELS
            )
            * PIXEL_AREA_KM2,
            3,
        )
    ).all()
)

if not all_checks_passed:
    raise ValueError(
        "One or more 2030 scenario checks failed."
    )

print("\nAll 2030 scenario checks passed.")
print("\nScenario summary saved:")
print(SCENARIO_SUMMARY_PATH)

,scenario,restored_pixels,restored_area_km2,baseline_core_area_km2,scenario_core_area_km2,invalid_changes,mean_distance_to_core_m,mean_distance_to_urban_m,mean_surrounding_core,mean_surrounding_urban,mean_suitability_score
0,dispersed,18649,11.656,54.407,66.063,0,371.65,115.62,0.0393,0.1094,0.4675
1,patch_enlargement,18649,11.656,54.407,66.063,0,31.11,149.93,0.1229,0.0806,0.6748
2,connectivity_focused,18649,11.656,54.407,66.063,0,119.53,142.50,0.2865,0.0787,0.7783
3,integrated_low_matrix_pressure,18649,11.656,54.407,66.063,0,123.22,255.78,0.2261,0.0539,0.8382



All 2030 scenario checks passed.

Scenario summary saved:
C:\Users\smit1\BlackGrouse_2030\outputs\tables\2030_restoration_scenario_summary.csv
